# Stratified split of synthetic and diffusion-augmented imagery.

Two image sources are combined into a single split:

  - Original synthetic images, distributed across train, val, and test by sequence according to SEQ_TO_SPLIT.
  - Diffusion-augmented images, assigned exclusively to train.

The original images alone yield a 20/40/40 train/val/test distribution by design; the augmented images then bring the train share up to a 60/20/20 image-level distribution overall.

Sequences are kept intact across splits — every image from a given sequence ends up in the same split as the rest — to prevent data leakage.

In [ ]:
import shutil
from pathlib import Path

BASE = Path(r"F:\Users\nikol\Desktop\Data\Pictures")

INPUT_ORIGINAL = {
    "t72": BASE / "t72syn",
    "t80": BASE / "t80syn",
    "t90": BASE / "t90syn",
}
INPUT_AUGMENTED = {
    "t72": BASE / "t72_augmented",
    "t80": BASE / "t80_augmented",
    "t90": BASE / "t90_augmented",
}
SUBFOLDERS = ["A1s", "A2s", "A3s", "A4s"]
OUTPUT_DIR = Path(r"F:\Users\nikol\Desktop\Data\Splits2")

# Original images: 320 train / 640 val / 640 test before augmentation is added.
SEQ_TO_SPLIT = {
    "A1_D1_LN_B2_C2": "train",
    "A2_D1_LN_B2_C2": "train",
    "A1_D1_L1_B1_C1": "val",
    "A1_D1_L2_B1_C1": "val",
    "A1_D1_L3_B1_C1": "val",
    "A1_D1_L4_B1_C1": "val",
    "A3_D1_L3_B1_C1": "val",
    "A3_D1_L4_B1_C1": "val",
    "A4_D1_LN_B2_C2": "val",
    "A4_D1_LN_B2_C1": "val",
    "A4_D1_L1_B1_C1": "val",
    "A4_D1_L2_B1_C1": "val",


    "A2_D1_L1_B1_C1": "test",
    "A2_D1_L2_B1_C1": "test",
    "A2_D1_L3_B1_C1": "test",
    "A2_D1_L4_B1_C1": "test",
    "A3_D1_L1_B1_C1": "test",
    "A3_D1_L2_B1_C1": "test",
    "A3_D1_LN_B2_C2": "test",
    "A4_D1_L3_B1_C1": "test",
    "A4_D1_L4_B1_C1": "test",
}


def get_images(folder):
    """Return all JPG/JPEG filenames in folder."""
    return [f.name for f in folder.iterdir() if f.suffix.lower() in (".jpg", ".jpeg")]


def parse_sequence_key_original(filename):
    """Extract sequence key from original synthetic filename: parts 0–4."""
    parts = Path(filename).stem.split("_")
    return "_".join(parts[:5])


def copy_file(src, dst_folder, new_name=None):
    dst_folder.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst_folder / (new_name or src.name))


def main():
    summary = []
    total_train = total_val = total_test = 0

    for class_name in INPUT_ORIGINAL:
        class_counts = {"train": 0, "val": 0, "test": 0}
        unknown_keys = set()

        # Original images: route each by sequence key into its target split.
        for sub in SUBFOLDERS:
            folder = INPUT_ORIGINAL[class_name] / sub
            if not folder.exists():
                continue

            for f in get_images(folder):
                key = parse_sequence_key_original(f)
                split = SEQ_TO_SPLIT.get(key)
                if split is None:
                    unknown_keys.add(key)
                    continue
                copy_file(folder / f, OUTPUT_DIR / split / class_name, f"{class_name}_{sub}_{f}")
                class_counts[split] += 1

        # Augmented images: always go into train.
        aug_folder = INPUT_AUGMENTED[class_name]
        aug_count = 0
        if aug_folder.exists():
            for f in get_images(aug_folder):
                copy_file(aug_folder / f, OUTPUT_DIR / "train" / class_name)
                aug_count += 1
        class_counts["train"] += aug_count

        if unknown_keys:
            print(f"  WARNING {class_name}: unknown sequences: {unknown_keys}")

        class_total = sum(class_counts.values())
        print(
            f"{class_name}: {class_counts['train']} train ({aug_count} aug) | "
            f"{class_counts['val']} val | {class_counts['test']} test"
        )

        if class_total > 0:
            summary.append(
                f"{class_name}: {class_counts['train']} train, {class_counts['val']} val, "
                f"{class_counts['test']} test "
                f"({class_counts['train']/class_total:.1%} / "
                f"{class_counts['val']/class_total:.1%} / "
                f"{class_counts['test']/class_total:.1%})"
            )
        total_train += class_counts["train"]
        total_val += class_counts["val"]
        total_test += class_counts["test"]

    print("\nDistribution:")
    for line in summary:
        print(" -", line)

    overall = total_train + total_val + total_test
    if overall > 0:
        print(
            f"\nOverall: {total_train} train, {total_val} val, {total_test} test "
            f"({total_train/overall:.1%} / {total_val/overall:.1%} / {total_test/overall:.1%})"
        )


if __name__ == "__main__":
    main()

t72: 1920 train (1600 aug) | 640 val | 640 test
t80: 1920 train (1600 aug) | 640 val | 640 test
t90: 1920 train (1600 aug) | 640 val | 640 test

=== Distribution ===
 - t72: 1920 train, 640 val, 640 test (60.0% / 20.0% / 20.0%)
 - t80: 1920 train, 640 val, 640 test (60.0% / 20.0% / 20.0%)
 - t90: 1920 train, 640 val, 640 test (60.0% / 20.0% / 20.0%)

Overall: 5760 train, 1920 val, 1920 test (60.0% / 20.0% / 20.0%)
